# dscraft.forecast quickstart

This notebook demonstrates `dscraft.forecast`'s classical statistical forecasting (AutoARIMA + AutoETS via Nixtla's `statsforecast`) over a Tier-1 Arrow-backed pipeline.

**Section 1** forecasts and backtests a synthetic seasonal (sine-wave-plus-trend) multi-series panel.

**Section 2** runs the exact same public API (`forecast()`/`backtest()`, same `ForecastConfig` shape) against a **real** dataset -- Mauna Loa atmospheric CO2 concentration, resampled to monthly means -- bundled inside `statsmodels` (no network access).

This mirrors `packages/dscraft/examples/forecast/forecast_example.py` in notebook form.

Requires the `forecast` and `dev` extras (the latter for `statsmodels`, used only to supply the real bundled CO2 dataset):
```bash
pip install -e "packages/dscraft[forecast,dev]"
```

In [1]:
import warnings

# tqdm.auto emits a benign TqdmWarning at import time (via statsforecast's
# own import of tqdm.auto) when ipywidgets isn't installed -- it just means
# tqdm falls back to its plain-text progress bar instead of a widget-based
# one; it does not affect forecast()/backtest() correctness. Suppressed
# narrowly (this category only) rather than broadly, matching
# dscraft.forecast.forecast.py's own narrowly-scoped
# `warnings.simplefilter("ignore", category=UserWarning)` pattern around
# its statsforecast model-fit calls.
from tqdm.std import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

import numpy as np
import pandas as pd

from dscraft.forecast import ForecastConfig, backtest, forecast, validate_input


def build_synthetic_seasonal_panel(n_points: int = 150, seed: int = 7) -> pd.DataFrame:
    """Two seasonal-plus-trend series with a fixed seed -- a demo fixture only."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2024-01-01", periods=n_points, freq="D")
    t = np.arange(n_points)

    series_params = {
        "store_1_sales": {"trend": 0.08, "amplitude": 12.0, "phase": 0.0, "level": 200.0},
        "store_2_sales": {"trend": -0.04, "amplitude": 6.0, "phase": 2.0, "level": 120.0},
    }
    frames = []
    for unique_id, params in series_params.items():
        seasonal = params["amplitude"] * np.sin(2 * np.pi * t / 7 + params["phase"])
        trend = params["trend"] * t
        noise = rng.normal(scale=1.0, size=n_points)
        y = params["level"] + trend + seasonal + noise
        frames.append(pd.DataFrame({"unique_id": unique_id, "ds": dates, "y": y}))

    panel = pd.concat(frames, ignore_index=True)
    # Convert to Tier-1 Arrow-backed pandas (pandas 2.x ArrowDtype).
    return panel.convert_dtypes(dtype_backend="pyarrow")


panel = build_synthetic_seasonal_panel()

print("=== Tier-1 input validation ===")
report = validate_input(panel)
print(f"input_kind={report.input_kind} n_rows={report.n_rows} n_series={report.n_series}")
print(f"is_fully_arrow_backed={report.is_fully_arrow_backed}")

=== Tier-1 input validation ===
input_kind=pandas n_rows=300 n_series=2
is_fully_arrow_backed=True


## Section 1: forecast + backtest the synthetic panel

In [2]:
config = ForecastConfig(
    horizon=14,
    freq="D",
    season_length=7,
    models=("AutoARIMA", "AutoETS"),
)

print("=== Forecast (next 14 days) ===")
forecasts = forecast(panel, config)
print(forecasts.head(10).to_string(index=False))

print("\n=== Backtest (holding out the last 14 days per series) ===")
synthetic_result = backtest(panel, config, test_size=14)
print(synthetic_result.to_frame().to_string(index=False))

print("\n=== Section 1 summary (synthetic) ===")
for model_name in config.models:
    print(
        f"{model_name}: mean MAE = {synthetic_result.mean_mae(model_name):.3f}, "
        f"mean RMSE = {synthetic_result.mean_rmse(model_name):.3f}"
    )
print(
    f"Overall: mean MAE = {synthetic_result.mean_mae():.3f}, "
    f"mean RMSE = {synthetic_result.mean_rmse():.3f}"
)

=== Forecast (next 14 days) ===


    unique_id         ds  AutoARIMA    AutoETS
store_1_sales 2024-05-30 217.020304 217.056842
store_1_sales 2024-05-31 206.813577 206.960414
store_1_sales 2024-06-01 200.314186 200.266212
store_1_sales 2024-06-02 203.011189 202.948982
store_1_sales 2024-06-03 212.071272 212.252440
store_1_sales 2024-06-04 221.872807 221.816543
store_1_sales 2024-06-05 224.314823 224.351794
store_1_sales 2024-06-06 217.592653 217.629172
store_1_sales 2024-06-07 207.385926 207.532744
store_1_sales 2024-06-08 200.886535 200.838542

=== Backtest (holding out the last 14 days per series) ===


    unique_id     model      mae     rmse  n_points  expected_points
store_1_sales AutoARIMA 0.653120 0.809875        14               14
store_1_sales   AutoETS 0.605170 0.747666        14               14
store_2_sales AutoARIMA 0.725755 0.919449        14               14
store_2_sales   AutoETS 0.720238 0.947082        14               14

=== Section 1 summary (synthetic) ===
AutoARIMA: mean MAE = 0.689, mean RMSE = 0.865
AutoETS: mean MAE = 0.663, mean RMSE = 0.847
Overall: mean MAE = 0.676, mean RMSE = 0.856


## Section 2: the same API against a real dataset (statsmodels co2, monthly)

In [3]:
from statsmodels.datasets import co2

raw = co2.load_pandas().data["co2"]
# The raw weekly series has scattered missing weeks (real-world equipment
# downtime); forecast()'s internal prepare_frame() deliberately rejects
# NaN/inf rather than imputing, so we interpolate here, caller-side.
monthly = raw.resample("MS").mean().interpolate()
co2_panel = pd.DataFrame({"unique_id": "co2_monthly", "ds": monthly.index, "y": monthly.to_numpy()})

co2_config = ForecastConfig(horizon=12, freq="MS", season_length=12, models=("AutoARIMA", "AutoETS"))

print("=== Tier-1 input validation ===")
co2_report = validate_input(co2_panel)
print(f"input_kind={co2_report.input_kind} n_rows={co2_report.n_rows} n_series={co2_report.n_series}")

print("\n=== Forecast (next 12 months) ===")
co2_forecasts = forecast(co2_panel, co2_config)
print(co2_forecasts.to_string(index=False))

print("\n=== Backtest (holding out the last 12 months) ===")
co2_result = backtest(co2_panel, co2_config, test_size=12)
print(co2_result.to_frame().to_string(index=False))

print("\n=== Section 2 summary (real co2 dataset) ===")
for model_name in co2_config.models:
    print(
        f"{model_name}: mean MAE = {co2_result.mean_mae(model_name):.3f}, "
        f"mean RMSE = {co2_result.mean_rmse(model_name):.3f}"
    )
print(f"Overall: mean MAE = {co2_result.mean_mae():.3f}, mean RMSE = {co2_result.mean_rmse():.3f}")

=== Tier-1 input validation ===
input_kind=pandas n_rows=526 n_series=1

=== Forecast (next 12 months) ===


  unique_id         ds  AutoARIMA    AutoETS
co2_monthly 2002-01-01 371.992561 371.842636
co2_monthly 2002-02-01 372.832212 372.674638
co2_monthly 2002-03-01 373.719743 373.543083
co2_monthly 2002-04-01 374.932730 374.664203
co2_monthly 2002-05-01 375.418504 375.219143
co2_monthly 2002-06-01 374.782222 374.699542
co2_monthly 2002-07-01 373.317102 373.180355
co2_monthly 2002-08-01 371.261898 371.186614
co2_monthly 2002-09-01 369.483424 369.475875
co2_monthly 2002-10-01 369.751318 369.695997
co2_monthly 2002-11-01 371.123659 371.052237
co2_monthly 2002-12-01 372.567295 372.496628

=== Backtest (holding out the last 12 months) ===


  unique_id     model      mae     rmse  n_points  expected_points
co2_monthly AutoARIMA 0.332201 0.382972        12               12
co2_monthly   AutoETS 0.198246 0.250281        12               12

=== Section 2 summary (real co2 dataset) ===
AutoARIMA: mean MAE = 0.332, mean RMSE = 0.383
AutoETS: mean MAE = 0.198, mean RMSE = 0.250
Overall: mean MAE = 0.265, mean RMSE = 0.317


## Side-by-side: synthetic vs. real backtest error

In [4]:
print(f"{'dataset':<20}{'mean MAE':>12}{'mean RMSE':>12}")
print(f"{'synthetic (sine)':<20}{synthetic_result.mean_mae():>12.3f}{synthetic_result.mean_rmse():>12.3f}")
print(f"{'real (co2, monthly)':<20}{co2_result.mean_mae():>12.3f}{co2_result.mean_rmse():>12.3f}")

dataset                 mean MAE   mean RMSE
synthetic (sine)           0.676       0.856
real (co2, monthly)        0.265       0.317


## Summary

This notebook forecasted and backtested a synthetic seasonal panel, then repeated the exact same `dscraft.forecast` API against a real bundled dataset (Mauna Loa atmospheric CO2), comparing backtest error side-by-side.

See `packages/dscraft/README.md`'s `## dscraft.forecast` section for more detail on what's implemented (classical AutoARIMA/AutoETS) and deferred (tree-based ML, zero-shot TSFMs, conformal prediction).